## setup

In [ ]:
# import stuff
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# load data from folder
folder = Path('../data')

# filtered dataset with mortgage rates
sold = pd.read_csv(folder / 'sold_with_rates.csv', low_memory = False)

# null count summary as reference for cleaning
sold_null_summary = pd.read_csv(folder / 'sold_null_summary.csv', index_col = 0)

In [ ]:
# ensure datasets loaded properly
print('Sold dataset shape:', sold.shape)
sold.head()

In [ ]:
sold_null_summary.head()

## data cleaning & prep
- convert date fields to datetime format
- remove unnecessary/redundant cols
- handle missing values appropriately
- ensure numeric fields are properly typed
- remove/flag invalid numeric values

In [ ]:
# convert date columns to datetime format
date_cols = ['CloseDate',
             'PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']
sold[date_cols] = sold[date_cols].apply(pd.to_datetime, errors = 'coerce')

# check that changes have been made
sold[date_cols].dtypes

In [ ]:
# check cols w >90% nulls
flag_over_90 = sold_null_summary[sold_null_summary['null pct'] > 90].index.tolist()
flag_over_90.sort()
print(len(flag_over_90))
flag_over_90

From the ``Real_Estate_Primer.pdf``, recall that our key data fields are:
- ``ListingKey``
- ``ListingContractDate``
- ``ListPrice``
- ``ClosePrice``
- ``PurchaseContractDate``
- ``CloseDate``
- ``LivingArea``
- ``BedroomsTotal``
- ``BathroomsTotalInteger``
- ``Latitude``/``Longitude``
- ``UnparsedAddress``

Since the above columns are not part of key fields, we can remove them.

In [ ]:
# drop cols w >90% nulls
sold_clean = sold.drop(columns = flag_over_90)

print('Sold shape after dropping:', sold_clean.shape)
sold_clean.head()

### shape after dropping >90% null: (448253, 69)

We can also consider dropping columns with over 50% nulls for more meaningful analyses, but we will still make sure that none of these flagged columns are considered key data fields. From our Week 2-3 deliverable instructions, we were given a list of key numeric fields (in which I will define as ``core_fields``), so we should also make sure that they are not flagged.

In [ ]:
# consider dropping cols w >50% nulls
flag_over_50 = sold_null_summary[sold_null_summary['null pct'] > 50].index.tolist()

# recall wk2-3: remove core fields from the list of cols to drop
core_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
               'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
               'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

for field in core_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields)') # overlaps with >90% null cols
flag_over_50

In week 6, we will be feature engineering using existing columns and also adding school districts using properties' latitude and longitude values, so we will exclude removing schools and school districts in case we end up populating them in future deliverables.

In [ ]:
# remove schools and school districts from flagged cols
school_fields = ['ElementarySchool',
                 'ElementarySchoolDistrict',
                 'MiddleOrJuniorSchool',
                 'MiddleOrJuniorSchoolDistrict',
                 'HighSchool']

for field in school_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

for i in flag_over_90:
    for j in flag_over_50:
        if j in flag_over_90:
            flag_over_50.remove(j)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields & schools):')
flag_over_50

In [ ]:
# drop cols w >50% nulls (excl. core fields and schools)
sold_clean = sold_clean.drop(columns = flag_over_50)
print('Sold shape after dropping:', sold_clean.shape)

sold_clean.head()

### shape after dropping >50% null: (448253, 60)

We can take a look at the remaining columns and determine what else to remove.

In [ ]:
sorted(sold_clean.columns)

In [ ]:
# as we investigate the remaining columns, we can set up a list and add onto it
cols_to_remove = []

sold[['ListAgentFirstName', 'ListAgentFullName', 'ListAgentLastName', 'ListAgentEmail']].isna().sum()

In [ ]:
sold['ListAgentFullName'].value_counts()

# we can remove list agent first name and last name since there's already a fullname col
cols_to_remove.append('ListAgentFirstName')
cols_to_remove.append('ListAgentLastName')

# not needed for analysis
cols_to_remove.append('ListAgentEmail')

In [ ]:
# street numbers are unique + not needed for analysis
print(sold['StreetNumberNumeric'].value_counts())

cols_to_remove.append('StreetNumberNumeric')

In [ ]:
sold[['ListingId', 'ListingKey', 'ListingKeyNumeric']].value_counts()

'''
ListingId   ListingKey  ListingKeyNumeric
SR25229893  1137564166  1137564166           3
CV25004341  1101502229  1101502229           3
OR24107765  1075843364  1075843364           3
MB24118151  1076326017  1076326017           3
OR24078258  1069697643  1069697643           2
                                            ..
CV25099944  1112966360  1112966360           1
CV25099932  1112687525  1112687525           1
CV25099884  1112680216  1112680216           1
CV25099848  1112869126  1112869126           1
WS26138647  1176037656  1176037656           1
Name: count, Length: 447914, dtype: int64
'''
sold_clean['ListingKeyNumeric'].equals(sold_clean['ListingKey'])
# True

cols_to_remove.append('ListingKeyNumeric') # since values are the same as listingkey

In [ ]:
print(sold_clean[['PropertyType', 'PropertySubType']].value_counts())

cols_to_remove.append('PropertyType')

In [ ]:
print(sold_clean[['LotSizeAcres', 'LotSizeArea', 'LotSizeSquareFeet']])

'''
        LotSizeAcres  LotSizeArea  LotSizeSquareFeet
0                NaN          NaN                NaN
1                NaN          NaN                NaN
2                NaN          NaN                NaN
3             0.2576     11219.00            11219.0
4             1.7100         1.71            74487.6    # area rep in acres
...              ...          ...                ...
448248        0.1631      7103.00             7103.0    # sq.ft
448249        0.6939     30228.00            30228.0
448250        0.1008      4389.00             4389.0    # area rep in sq.ft
448251        2.6124    113797.00           113797.0
448252     2452.6300      2452.63        106836562.8    # acres

[448253 rows x 3 columns]
'''

# units are not consistent in this column, values are either represented in acres or in square feet
cols_to_remove.append('LotSizeArea')

In [ ]:
sold_clean['StateOrProvince'].value_counts()

In [ ]:
sold_clean['StateOrProvince'].unique()

In [ ]:
# investigate nan value
sold_clean[sold_clean['StateOrProvince'].isnull()][['Latitude', 'Longitude', 'StateOrProvince']]

In [ ]:
''' 
cali coords
lat: (32.0, 42.5)
lon: (-125.0, -113.5)
'''
# since there's only 1 nan value, we can fill it with cali coords since lat/lon are within cali
sold_clean['StateOrProvince'] = sold_clean['StateOrProvince'].fillna('CA')
print(sold_clean.iloc[435015]['StateOrProvince'])

# double check that the nan value is gone
print('Double check:', sold_clean['StateOrProvince'].unique())

# since all properties are supposedly in cali, we dont need this column
cols_to_remove.append('StateOrProvince')

In [ ]:
sold_clean[['MLSAreaMajor', 'MlsStatus', 'BuyerAgentMlsId']]

# sold_clean['MlsStatus'].value_counts()
'''
MlsStatus
Closed    443797
Name: count, dtype: int64
'''

In [ ]:
# we remove these since BuyerAgentMlsId exists & we also wont be looking at the buyer's name
cols_to_remove.append('BuyerAgentFirstName')
cols_to_remove.append('BuyerAgentLastName')

### REMOVE COLUMNS

In [ ]:
# double check before removing
cols_to_remove

#### reviewing columns to remove

| column name | description |
| ----------- | ---------- |
| ListAgentFirstName | The first name of the listing agent |
| ListAgentLastName | The last name of the listing agent |
| StreetNumberNumeric | The integer portion of the street number |
| PropertyType | The list of types of properties such as Residential, Lease, Income, Land, Mobile, Commercial Sale, etc |
| ListAgentEmail | The email address of the Listing Agent |
| LotSizeArea | The total area of the lot. See Lot Size Units for the units of measurement (Square Feet, Square Meters, Acres, etc) |
| StateOrProvince | Text field containing the accepted postal abbreviation for the state or province | 
| ListingKeyNumeric | same as ``ListingKey`` column |
| BuyerAgentFirstName | The first name of the buyer's agent |
| BuyerAgentLastName | The last name of the buyer's agent |

In [ ]:
# drop columns
sold_clean = sold_clean.drop(columns = cols_to_remove)

In [ ]:
print(sold_clean.shape)
print(sold_clean.columns)

### final shape after dropping: (448253, 50)

### CLEAN ROWS

now that we've dealt with columns, lets look into invalid values for ClosePrice, LivingArea, DOM, and negative bedrooms or bathrooms

In [ ]:
# flag for negative closeprice
sold_clean['neg_closeprice_flag'] = sold_clean['ClosePrice'] <= 0

sold_clean[sold_clean['neg_closeprice_flag'] == True]

In [ ]:
# flag for negative livingarea
sold_clean['neg_livingarea_flag'] = sold_clean['LivingArea'] <= 0

sold_clean[sold_clean['neg_livingarea_flag'] == True]

In [ ]:
# look @ DOM
sold_clean[['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'DaysOnMarket']].head()

In [ ]:
# flag @ negative DOM
sold_clean['neg_dom_flag'] = sold_clean['DaysOnMarket'] < 0

sold_clean[sold_clean['neg_dom_flag'] == True].shape

In [ ]:
# look @ negative bedrooms
print(sold_clean[sold_clean['BedroomsTotal'] < 0].shape)
sold_clean[sold_clean['BedroomsTotal'] < 0].head()

In [ ]:
# look @ negative bathrooms
sold_clean[sold_clean['BathroomsTotalInteger'] < 0]

### data consistency checks
- validate logical order of date fields (ListingContractDate < PurchaseContractDate < CloseDate)
- create boolean flag cols to mark records that violate these rules
    - listing_after_close_flag
    - purchase_after_close_flag
    - negative_timeline_flag

In [ ]:
# validate order of datefields
invalid_rows = sold_clean[~((sold_clean['ListingContractDate'] < sold_clean['PurchaseContractDate']) & 
                      (sold_clean['PurchaseContractDate'] < sold_clean['CloseDate']))]

print('Shape of rows where date fields are out of order', invalid_rows.shape)
invalid_rows[['ListingContractDate', 'PurchaseContractDate', 'CloseDate']].head()

In [ ]:
# create bool flag cols to mark records that violate these rules
# correct order: list date < purchase date < close date

# list date after close date
sold_clean['listing_after_close_flag'] = sold_clean['ListingContractDate'] > sold_clean['CloseDate']
print('# rows where list date is after close date:', sold_clean[sold_clean['listing_after_close_flag'] == True].shape[0])

# purchase date after close date
sold_clean['purchase_after_close_flag'] = sold_clean['PurchaseContractDate'] > sold_clean['CloseDate']
print('# rows where purchase date is after close date:', sold_clean[sold_clean['purchase_after_close_flag'] == True].shape[0])

# violates order
sold_clean['negative_timeline_flag'] = ~((sold_clean['ListingContractDate'] < sold_clean['PurchaseContractDate']) & 
                                         (sold_clean['PurchaseContractDate'] < sold_clean['CloseDate']))
print('# rows with illogical timeline:', sold_clean[sold_clean['negative_timeline_flag'] == True].shape[0])

# check that these columns were made
sold_clean[['listing_after_close_flag', 'purchase_after_close_flag', 'negative_timeline_flag']].head()

### geographic data checks
- flag records w missing coords (lat/lon is null)
- flag lat = 0 or lon = 0 (sentinel null vals)
- flag lon > 0 errors (cal coords should be negative)
- flag out-of-state/implausible coords

In [ ]:
# flag missing coords (lat/lon is null)
sold_clean['missing_coords_flag'] = (sold_clean['Latitude'].isna()) | (sold_clean['Longitude'].isna())

print(sold_clean[sold_clean['missing_coords_flag'] == True].shape)
sold_clean[sold_clean['missing_coords_flag'] == True].head()

In [ ]:
# lat = 0 or lon = 0
sold_clean['sentinel_coords_flag'] = (sold_clean['Latitude'] == 0) | (sold_clean['Longitude'] == 0)

print(sold_clean[sold_clean['sentinel_coords_flag'] == True].shape)
sold_clean[sold_clean['sentinel_coords_flag'] == True].head()

In [ ]:
# lon > 0 errors (cali coords should be negative)
sold_clean['pos_lon_flag'] = sold_clean['Longitude'] > 0

print(sold_clean[sold_clean['pos_lon_flag'] == True].shape)
sold_clean[sold_clean['pos_lon_flag'] == True].head()

In [ ]:
# out of state (oos) or implausible coords

'''cali coords range
lat: (32.0, 42.5)
lon: (-125.0, -113.5)
'''

# filter for only cali listings
sold_clean['oos_coords_flag'] = ~(sold_clean['Latitude'].between(32.0, 42.5) & sold_clean['Longitude'].between(-125.0, -113.5))

sold_clean['oos_coords_flag'].value_counts()

In [ ]:
print('# rows with missing coordinates:', sold_clean[sold_clean['missing_coords_flag'] == True].shape[0])
print('# rows with sentinel coordinates:', sold_clean[sold_clean['sentinel_coords_flag'] == True].shape[0])

print('# rows with non-California values (aka positive longitude):', sold_clean[sold_clean['pos_lon_flag'] == True].shape[0])
print('# rows with out of state/implausible coordinates:', sold_clean[sold_clean['oos_coords_flag'] == True].shape[0])

### review flagged cols

We created about 10 flagged columns:
- neg_closeprice_flag
- neg_livingarea_flag
- neg_dom_flag
- listing_after_close_flag
- purchase_after_close_flag
- negative_timeline_flag
- missing_coords_flag
- sentinel_coords_flag
- pos_lon_flag
- oos_coords_flag

So now we can start cleaning by rows

In [ ]:
sold_clean.iloc[:,-10:]

In [ ]:
sold_clean[sold_clean['neg_closeprice_flag'] == True]

In [ ]:
# remove the singular 0-value closeprice
sold_clean = sold_clean[sold_clean['neg_closeprice_flag'] == False]

print(sold_clean.shape)
sold_clean.head(3)

In [ ]:
# remove 0-value livingarea
sold_clean[sold_clean['neg_livingarea_flag'] == True]['LivingArea'].value_counts()

# LivingArea
# 0.0    165
# Name: count, dtype: int64

# remove 0-value livingarea
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[sold_clean['neg_livingarea_flag'] == False]

print('Shape after removing:', sold_clean.shape)
sold_clean.head(3)

In [ ]:
# fix neg days on market values
mask = sold_clean[sold_clean['neg_dom_flag'] == True]

print('# of rows with negative days on market:', mask[['ListingContractDate', 'CloseDate', 'DaysOnMarket']].shape[0])
mask[['ListingContractDate', 'CloseDate', 'DaysOnMarket']].head()

In [ ]:
# attempt to fix negative dom
sold_clean.loc[sold_clean['neg_dom_flag'] == True, 'DaysOnMarket'] = (sold_clean['CloseDate'] - sold_clean['ListingContractDate']).dt.days

mask[['ListingContractDate', 'CloseDate', 'DaysOnMarket']].head(3)

In [ ]:
# even after attempting to fix the negative dom, the values remain unchanged, so this could be
# because close dates were inputted before listing dates, so we can set these 51 entries to nulls
# since it makes up a small amount of our data
print('# rows with negative DOM:', sold_clean[sold_clean['neg_dom_flag'] == True]['DaysOnMarket'].shape)

print('# nulls before conversion:', sold_clean['DaysOnMarket'].isna().sum())
sold_clean.loc[sold_clean['neg_dom_flag'] == True, 'DaysOnMarket'] = np.nan

# check that negatives were converted to nulls
print('# nulls:', sold_clean['DaysOnMarket'].isna().sum())

# previous attempt: remove negative dom rows
# sold_clean = sold_clean[sold_clean['neg_dom_flag'] == False]

print('# nulls after conversion:', sold_clean['DaysOnMarket'].isna().sum())

In [ ]:
print('# of rows where listing date after close date:', sold_clean[sold_clean['listing_after_close_flag'] == True].shape[0])
sold_clean[sold_clean['listing_after_close_flag'] == True][['ListingContractDate', 'CloseDate', 'DaysOnMarket']]

In [ ]:
sold_clean[(sold_clean['listing_after_close_flag'] == True) & (sold_clean['DaysOnMarket'] > 0)][['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'DaysOnMarket']].head(3)

In [ ]:
# idea: use days on market to calculate the correct listing date
# actually i cant bc some are straight up out of order so maybe i should just null these rows?

# convert listing dates after close date into null
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['listing_after_close_flag'] == True, ['ListingContractDate', 'CloseDate']] = np.nan

# sold_clean = sold_clean[sold_clean['listing_after_close_flag'] == False]

print('Shape after conversion:', sold_clean.shape)
print(sold_clean[['ListingContractDate', 'CloseDate']].isna().sum())

In [ ]:
# investigate purchase date after close date
mask = sold_clean[sold_clean['purchase_after_close_flag'] == True]
print(mask.shape)

mask[['ListingContractDate', 'CloseDate', 'PurchaseContractDate']].head(3)

In [ ]:
# there isnt really a way to correct these so they can be converted to null
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['purchase_after_close_flag'] == True, ['ListingContractDate', 'CloseDate']] = np.nan

print('Shape after conversion:', sold_clean.shape)
print(sold_clean[['ListingContractDate', 'CloseDate']].isna().sum())

In [ ]:
# investigate negative timeline flag
mask = sold_clean[sold_clean['negative_timeline_flag'] == True]
print(mask.shape)

mask[['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'DaysOnMarket']].head(3)

In [ ]:
# as before, there isnt a way to fix the wrong dates, so we can turn them into null
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['negative_timeline_flag'] == True, ['ListingContractDate', 'CloseDate']] = np.nan

print('Shape after conversion:', sold_clean.shape)
print(sold_clean[['ListingContractDate', 'CloseDate']].isna().sum())


In [ ]:
# investigate missing coords flag
mask = sold_clean[sold_clean['missing_coords_flag'] == True]
print('# rows with missing coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

In [ ]:
# there isn't a way to correct lat/lon values but we can keep them
# since we can use other information in the row for analysis

# initial plan below: remove missing

# print('Shape before removing:', sold_clean.shape)
# sold_clean = sold_clean[sold_clean['missing_coords_flag'] == False]

# print('Shape after removing:', sold_clean.shape)

In [ ]:
# investigate sentinel coords
mask = sold_clean[sold_clean['sentinel_coords_flag'] == True]
print('# rows with sentinel coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

In [ ]:
# remove 0 lat/lon values
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[sold_clean['sentinel_coords_flag'] == False]

print('Shape after removing:', sold_clean.shape)

In [ ]:
# investigate positive longitude flag
mask = sold_clean[sold_clean['pos_lon_flag'] == True]
print('# rows outside CA:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# set coordinate values to null
# even though cities correspond to their zipcodes, the lat/lon aren't within CA bounds
print('Shape before conversion:', sold_clean[sold_clean['pos_lon_flag'] == True].shape)
sold_clean.loc[sold_clean['pos_lon_flag'] == True, ['Latitude', 'Longitude']] = np.nan

print('Shape after conversion:', sold_clean[sold_clean['pos_lon_flag'] == True].shape)
sold_clean[sold_clean['pos_lon_flag'] == True][['Latitude', 'Longitude']].head()

In [ ]:
# mask['City'].unique()

# array(['Santa Ana', 'Riverside', 'Spring Valley', 'Merced', 'Victorville',
#        'Carson', 'San Luis Obispo', 'Los Angeles', 'Hesperia',
#        'Fountain Valley', 'East Los Angeles', 'Orland', 'San Bernardino',
#        'Beaumont', 'Julian', 'Hawthorne', 'Palos Verdes Estates',
#        'Visalia', 'Fresno', 'Pico Rivera', 'Santa Maria', 'Lake Elsinore',
#        'Moreno Valley', 'Lancaster', 'Helendale'], dtype=object)



# mask['PostalCode'].unique()
# 
# array(['92703', '92504', '91977', '95340', '92392', '90746', '93401',
#        '90002', '92345', '92708', '90063', '95963', '92404', '90011',
#        '92395', '92223', '92036', '90250', '90274', '93291', '93730',
#        '90060', '93455', '92532', '92553', '92344', '93534', '92503',
#        '92342'], dtype=object)

In [ ]:
# investigate out of state coords
mask = sold_clean[sold_clean['oos_coords_flag'] == True]
print('# rows with out of state/implausible coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
mask['City'].unique()

In [ ]:
# look into null cities
remove_mask = sold_clean[((sold_clean['oos_coords_flag'] == True) & (sold_clean['City'].isnull()))]
print(remove_mask.shape)
remove_mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
remove_mask['PostalCode'].unique()

In [ ]:
zip_to_city = {
    '91932': 'Imperial Beach',
    '93933': 'Marina',
    '94065': 'Redwood City',
    '92028': 'Fallbrook',
    '94904': 'Greenbrae',
    '95346': 'Mi Wuk Village',
    '92101': 'San Diego',
    '94574': 'St. Helena',
    '11217': np.nan,        # valid city but out of CA
    '63119-2447': np.nan,   # valid city but out of CA
    '88888': np.nan,        # valid city but out of CA
    '85374': np.nan         # valid city but out of CA
}

In [ ]:
# apply mapping
mask = (
    (sold_clean['oos_coords_flag']) &
    (sold_clean['City'].isna())
)

sold_clean.loc[mask, 'City'] = (
    sold_clean.loc[mask, 'PostalCode']
    .astype(str)
    .map(zip_to_city)
)

In [ ]:
# check that changes have been applied
sold_clean[mask][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# remove non-CA rows
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[~((sold_clean['oos_coords_flag'] == True) & (sold_clean['City'].isnull()))]

print('Shape after removing:', sold_clean.shape)

In [ ]:
# look into out of state
mask = sold_clean[(sold_clean['City'] == 'Outside Area (Outside Ca)') |
           (sold_clean['City'] == 'Other') |
           (sold_clean['City'] == 'Outside Area (Outside U.S.) Foreign Country')][['Latitude', 'Longitude', 'City', 'PostalCode']]
print(mask.shape)

mask

In [ ]:
# remove out of state coords except cities marked 'Other'

print('Shape before cleaning:', sold_clean.shape)
sold_clean = sold_clean[
    ~sold_clean['City'].isin([
        'Outside Area (Outside Ca)',
        'Outside Area (Outside U.S.) Foreign Country'
    ])
]

print('Shape after cleaning:', sold_clean.shape)

In [ ]:
# double check rows were removed
sold_clean[sold_clean['oos_coords_flag']]['City'].unique()

In [ ]:
# check postal codes within cali
sold_clean[
    (sold_clean['City'] == 'Other') &
    (sold_clean['PostalCode'].str.startswith('9'))
    ][['City', 'PostalCode']]

In [ ]:
# rename cities correctly according to zipcode
zip_to_city = {
    95493: 'Witter Springs',
    90032: 'Los Angeles',
    92109: 'San Diego',
    90063: 'Los Angeles',
    93603: 'Badger',
    95602: 'Auburn',
    90044: 'Los Angeles',
    90011: 'Los Angeles',
    93654: 'Reedley'
}

mask = sold_clean['City'] == 'Other'

# apply fixes
sold_clean.loc[mask, 'City'] = (
    sold_clean.loc[mask, 'PostalCode']
    .map(zip_to_city)
)

In [ ]:
print(sold_clean[sold_clean['oos_coords_flag']].shape)
sold_clean[sold_clean['oos_coords_flag']][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
print('Shape before conversion:', sold_clean.shape)
sold_clean.loc[sold_clean['oos_coords_flag'] == True, ['Latitude', 'Longitude']] = np.nan

print('Shape after conversion:', sold_clean.shape)
sold_clean[sold_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# look into nulls and flagstaff area
sold_clean[(sold_clean['oos_coords_flag'] == True) &
           ((sold_clean['City'].isna()) |
           (sold_clean['City'] == 'Flagstaff'))
           ][['City', 'PostalCode']]

In [ ]:
# remove out of state rows + row with flagstaff since that's in AZ
mask = (
    sold_clean['oos_coords_flag'] &
    (sold_clean['City'].isna() |
     (sold_clean['City'] == 'Flagstaff'))
)

print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean[~mask]

print('Shape after removing:', sold_clean.shape)

In [ ]:
sold_clean[sold_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']]

In [ ]:
# fix palmdale zipcode
sold_clean.loc[319267, 'PostalCode'] = 93551 # used to be 83551

In [ ]:
# normalize zipcodes to remove hyphens (e.g 63119-2447)
sold_clean['PostalCode'] = (
    sold_clean['PostalCode']
    .astype(str)
    .str[:5]
)

In [ ]:
# drop flagged columns that're done
print('Shape before removing:', sold_clean.shape)
sold_clean = sold_clean.drop(columns = ['neg_closeprice_flag',
                                        'neg_livingarea_flag',
                                        'neg_dom_flag',
                                        'listing_after_close_flag',
                                        'purchase_after_close_flag',
                                        'negative_timeline_flag',
                                        'missing_coords_flag',
                                        'sentinel_coords_flag',
                                        'pos_lon_flag',
                                        'oos_coords_flag'])

print('Shape after removing:', sold_clean.shape)

In [ ]:
sold_clean.columns

### deliverable
- document every xformation made & why
- include:
    - before/after counts
    - dtype confirmations
    - date consistency flag counts
    - geographic data quality summary noting invalid coordinate records

### ``SOLD`` TRANSFORMATION SUMMARY
<hr>

> Dataset shape: (448253, 84)

The following date fields were converted to datetime format:
- CloseDate
- PurchaseContractDate
- ListingContractDate
- ContractStatusChangeDate


In [ ]:
sold[date_cols].dtypes

The following 15 columns were identified to contain __over 90% nulls__ and were removed:
- AboveGradeFinishedArea
- BasementYN
- BelowGradeFinishedArea
- BuilderName
- BuildingAreaTotal
- BusinessType
- CoBuyerAgentFirstName
- CoveredSpaces
- ElementarySchoolDistrict
- FireplacesTotal
- LotSizeDimensions
- MiddleOrJuniorSchoolDistrict
- TaxAnnualAmount
- TaxYear
- WaterfrontYN

> Updated dataset shape: (448253, 69)

The following 9 columns were considered and removed for having __over 50% nulls__:
- AssociationFeeFrequency
- BuyerAgencyCompensation
- BuyerAgencyCompensationType
- CoListAgentFirstName
- CoListAgentLastName
- CoListOfficeName
- OriginatingSystemName
- OriginatingSystemSubName
- SubdivisionName

> Updated dataset shape: (448253, 60)

The remaining 60 columns were manually inspected and the following 10 were removed as they were considered to be unnecessary for analysis:
- ListAgentFirstName
- ListAgentLastName
- ListAgentEmail
- StreetNumberNumeric
- ListingKeyNumeric
- PropertyType
- LotSizeArea
- StateOrProvince
- BuyerAgentFirstName
- BuyerAgentLastName

> Updated dataset shape: (448253, 50)

Rows were also cleaned by flagging for invalid values. Below is the count for each flag, how it was handled, and how it affected the overall dataset shape.

#### VALIDITY CHECKS
<hr>

- neg_closeprice_flag
    - count: 1
    - handled by: removal
    - notes: 0-value
    > updated dataset shape: (448252, 60)
- neg_livingarea_flag
    - count: 165
    - handled by: removal
    - notes: all 0-values
    > updated dataset shape: (448087, 60)
- neg_dom_flag
    - count: 51
    - handled by: converted to null
    - notes: attempts to fix negative value didn't work, maybe because CloseDate was inputted before ListingContractDate, so they were set to null so we can still use other information given by the row
    > updated dataset shape: (448087, 60) || UNCHANGED

There were no rows with negative bedrooms or bathrooms, so flag columns were not created for them

#### CONSISTENCY CHECKS
<hr>

- listing_after_close_flag
    - count: 70
    - handled by: converted to null
    > updated dataset shape: (448087, 60) || UNCHANGED
- purchase_after_close_flag
    - count: 239
    - handled by: converted to null
    > updated dataset shape: (448087, 60) || UNCHANGED
- negative_timeline_flag
    - count: 14940
    - handled by: converted to null
    > updated dataset shape: (448087, 60) || UNCHANGED

### GEOGRAPHIC CHECKS
<hr>

- missing_coords_flag
    - count: 4385
    - handled by: 
    > updated dataset shape: (448087, 60) || UNCHANGED
- sentinel_coords_flag
    - count: 37
    - handled by: removal
    > updated dataset shape: (448050, 60)
- pos_lon_flag
    - count: 31
    - handled by: converted to null
    > updated dataset shape: (448050, 60) || UNCHANGED
- oos_coords_flag
    - count: 4448
    - handled by: removal
    - transformation 1: null cities were mapped according to zipcode
    > updated dataset shape: (448045, 60) || UNCHANGED
    - transformation 2: out of state coordinates removed
    > updated dataset shape: (448033, 60)
    - transformation 3: 'Other' cities were renamed according to zipcode
    - transformation 4: remove more out of state rows
    > updated dataset shape: (448026, 60)
    - transformation 5: manually fix palmdale zipcode
    - transformation 6: normalize zipcodes to remove hyphens

In the end, all 10 flagged columns were dropped which leaves us with:
## final dataset shape: (448026, 50)

> recall starting dataset shape (448253, 84)

Other thoughts: While majority of the dataset has been cleaned, a small percentage of data were converted to nulls, especially when it comes to geographic data. When visualizing geographic data, these null values will be excluded and noted in future reports or presentations. The reason they are kept in the dataset is because those rows contain other useful information that can be visualized with no problem (e.g comparing list price to living area of a property)

In [ ]:
sold_clean.isna().sum()

In [ ]:
# save dataset as csv
sold_clean.to_csv('../data/sold_clean.csv', index = False)